In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(r"D:\projects\f1-project")

laps_path = PROJECT_ROOT / "data" / "processed" / "laps_2023_2024_first5.csv"

laps_df = pd.read_csv(laps_path)

print("Rows:", len(laps_df))
print("Races:", laps_df[["season", "round"]].drop_duplicates().shape[0])
laps_df.head()

Rows: 10050
Races: 10


,season,round,lap,driver_id,position,lap_time
0,2023,1,1,max_verstappen,1,1:39.019
1,2023,1,1,leclerc,2,1:40.230
2,2023,1,1,perez,3,1:41.063
3,2023,1,1,sainz,4,1:41.659
4,2023,1,1,hamilton,5,1:42.288


In [4]:
import requests

url = "https://api.jolpi.ca/ergast/f1/2023/1/pitstops/"

params = {"limit": 100}

response = requests.get(url, params=params, timeout=30)

print("Status code:", response.status_code)

data = response.json()

data

Status code: 200


{'MRData': {'xmlns': '',
  'series': 'f1',
  'url': 'https://api.jolpi.ca/ergast/f1/2023/1/pitstops/',
  'limit': '100',
  'offset': '0',
  'total': '50',
  'RaceTable': {'season': '2023',
   'round': '1',
   'Races': [{'season': '2023',
     'round': '1',
     'url': 'https://en.wikipedia.org/wiki/2023_Bahrain_Grand_Prix',
     'raceName': 'Bahrain Grand Prix',
     'Circuit': {'circuitId': 'bahrain',
      'url': 'https://en.wikipedia.org/wiki/Bahrain_International_Circuit',
      'circuitName': 'Bahrain International Circuit',
      'Location': {'lat': '26.0325',
       'long': '50.5106',
       'locality': 'Sakhir',
       'country': 'Bahrain'}},
     'date': '2023-03-05',
     'time': '15:00:00Z',
     'PitStops': [{'driverId': 'gasly',
       'lap': '9',
       'stop': '1',
       'time': '18:18:56',
       'duration': '25.885'},
      {'driverId': 'norris',
       'lap': '10',
       'stop': '1',
       'time': '18:20:31',
       'duration': '32.766'},
      {'driverId': 'tsunod

In [5]:
race = data["MRData"]["RaceTable"]["Races"][0]

pit_stops = race["PitStops"]

print("Race:", race["raceName"])
print("Pit-stop records:", len(pit_stops))

pit_stops[:5]

Race: Bahrain Grand Prix
Pit-stop records: 50


[{'driverId': 'gasly',
  'lap': '9',
  'stop': '1',
  'time': '18:18:56',
  'duration': '25.885'},
 {'driverId': 'norris',
  'lap': '10',
  'stop': '1',
  'time': '18:20:31',
  'duration': '32.766'},
 {'driverId': 'tsunoda',
  'lap': '10',
  'stop': '1',
  'time': '18:20:34',
  'duration': '25.267'},
 {'driverId': 'bottas',
  'lap': '11',
  'stop': '1',
  'time': '18:22:06',
  'duration': '25.399'},
 {'driverId': 'albon',
  'lap': '11',
  'stop': '1',
  'time': '18:22:12',
  'duration': '25.054'}]

In [6]:
pit_stops_df = pd.DataFrame(pit_stops)

# Add race identifiers so records can be matched across races
pit_stops_df["season"] = int(race["season"])
pit_stops_df["round"] = int(race["round"])

# Convert numeric columns from strings to integers
pit_stops_df["lap"] = pit_stops_df["lap"].astype(int)
pit_stops_df["stop"] = pit_stops_df["stop"].astype(int)

# Convert pit-stop duration from string to numeric seconds
pit_stops_df["duration"] = pd.to_numeric(
    pit_stops_df["duration"],
    errors="coerce"
)

pit_stops_df.head()

,driverId,lap,stop,time,duration,season,round
0,gasly,9,1,18:18:56,25.885,2023,1
1,norris,10,1,18:20:31,32.766,2023,1
2,tsunoda,10,1,18:20:34,25.267,2023,1
3,bottas,11,1,18:22:06,25.399,2023,1
4,albon,11,1,18:22:12,25.054,2023,1


In [7]:
pit_stops_df.info()

print("\nMissing values:")
print(pit_stops_df.isna().sum())

print("\nDuplicate pit-stop records:")
print(
    pit_stops_df.duplicated(
        subset=["season", "round", "driverId", "stop"]
    ).sum()
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   driverId  50 non-null     object 
 1   lap       50 non-null     int64  
 2   stop      50 non-null     int64  
 3   time      50 non-null     object 
 4   duration  50 non-null     float64
 5   season    50 non-null     int64  
 6   round     50 non-null     int64  
dtypes: float64(1), int64(4), object(2)
memory usage: 2.9+ KB

Missing values:
driverId    0
lap         0
stop        0
time        0
duration    0
season      0
round       0
dtype: int64

Duplicate pit-stop records:
0


In [8]:
def fetch_race_pit_stops(season, round_num):
    url = f"https://api.jolpi.ca/ergast/f1/{season}/{round_num}/pitstops/"

    params = {"limit": 100}
    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()

    races = data["MRData"]["RaceTable"]["Races"]

    # Some race endpoints may return no race/pit-stop data
    if not races:
        return pd.DataFrame()

    race = races[0]
    pit_stops = race.get("PitStops", [])

    rows = []

    for stop in pit_stops:
        rows.append({
            "season": int(season),
            "round": int(round_num),
            "driver_id": stop["driverId"],
            "lap": int(stop["lap"]),
            "stop": int(stop["stop"]),
            "time": stop.get("time"),
            "duration": pd.to_numeric(stop.get("duration"), errors="coerce")
        })

    return pd.DataFrame(rows)

In [9]:
test_pits = fetch_race_pit_stops(2023, 1)

print("Rows:", len(test_pits))
test_pits.head()

Rows: 50


,season,round,driver_id,lap,stop,time,duration
0,2023,1,gasly,9,1,18:18:56,25.885
1,2023,1,norris,10,1,18:20:31,32.766
2,2023,1,tsunoda,10,1,18:20:34,25.267
3,2023,1,bottas,11,1,18:22:06,25.399
4,2023,1,albon,11,1,18:22:12,25.054


In [10]:
race_list = (
    laps_df[["season", "round"]]
    .drop_duplicates()
    .sort_values(["season", "round"])
    .itertuples(index=False, name=None)
)

pit_stop_frames = []

for season, round_num in race_list:
    race_pits = fetch_race_pit_stops(season, round_num)

    pit_stop_frames.append(race_pits)

    print(
        f"{season} Round {round_num}: "
        f"{len(race_pits)} pit-stop records"
    )

all_pit_stops_df = pd.concat(
    pit_stop_frames,
    ignore_index=True
)

print("\nTotal pit-stop records:", len(all_pit_stops_df))
print(
    "Races collected:",
    all_pit_stops_df[["season", "round"]]
    .drop_duplicates()
    .shape[0]
)

2023 Round 1: 50 pit-stop records
2023 Round 2: 24 pit-stop records
2023 Round 3: 65 pit-stop records
2023 Round 4: 23 pit-stop records
2023 Round 5: 20 pit-stop records
2024 Round 1: 43 pit-stop records
2024 Round 2: 19 pit-stop records
2024 Round 3: 36 pit-stop records
2024 Round 4: 54 pit-stop records
2024 Round 5: 40 pit-stop records

Total pit-stop records: 374
Races collected: 10


In [11]:
print("Shape:", all_pit_stops_df.shape)

print("\nMissing values:")
print(all_pit_stops_df.isna().sum())

print("\nDuplicate stop keys:")
print(
    all_pit_stops_df.duplicated(
        subset=["season", "round", "driver_id", "stop"]
    ).sum()
)

print("\nUnique races:")
print(
    all_pit_stops_df[["season", "round"]]
    .drop_duplicates()
    .shape[0]
)

print("\nDuration summary (seconds):")
all_pit_stops_df["duration"].describe()

Shape: (374, 7)

Missing values:
season        0
round         0
driver_id     0
lap           0
stop          0
time          0
duration     70
dtype: int64

Duplicate stop keys:
0

Unique races:
10

Duration summary (seconds):


count    304.000000
mean      23.530148
std        4.561175
min       17.222000
25%       21.144000
50%       23.152000
75%       24.795500
max       48.938000
Name: duration, dtype: float64

In [12]:
missing_duration_by_race = (
    all_pit_stops_df
    .assign(duration_missing=all_pit_stops_df["duration"].isna())
    .groupby(["season", "round"])
    .agg(
        pit_stop_records=("driver_id", "size"),
        missing_durations=("duration_missing", "sum")
    )
    .reset_index()
)

missing_duration_by_race

,season,round,pit_stop_records,missing_durations
0,2023,1,50,0
1,2023,2,24,0
2,2023,3,65,49
3,2023,4,23,0
4,2023,5,20,0
5,2024,1,43,1
6,2024,2,19,1
7,2024,3,36,0
8,2024,4,54,18
9,2024,5,40,1


In [13]:
missing_duration_df = all_pit_stops_df[
    all_pit_stops_df["duration"].isna()
].copy()

missing_duration_df[
    ["season", "round", "driver_id", "lap", "stop", "time", "duration"]
].head(20)

,season,round,driver_id,lap,stop,time,duration
84,2023,3,hamilton,8,1,15:17:43,NaN
85,2023,3,max_verstappen,8,1,15:17:45,NaN
86,2023,3,alonso,8,1,15:17:48,NaN
87,2023,3,stroll,8,1,15:17:51,NaN
88,2023,3,gasly,8,1,15:17:53,NaN
89,2023,3,hulkenberg,8,1,15:17:55,NaN
90,2023,3,russell,8,2,15:17:59,NaN
91,2023,3,tsunoda,8,1,15:18:02,NaN
92,2023,3,norris,8,1,15:18:04,NaN
93,2023,3,piastri,8,1,15:18:06,NaN


In [14]:
missing_duration_df.groupby(["season", "round"]).size()

season  round
2023    3        49
2024    1         1
        2         1
        4        18
        5         1
dtype: int64

In [15]:
# Create the processed-data folder if it doesn't exist
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

pit_stops_path = processed_dir / "pit_stops_2023_2024_first5.csv"

all_pit_stops_df.to_csv(pit_stops_path, index=False)

print("Saved to:", pit_stops_path)
print("Rows saved:", len(all_pit_stops_df))

Saved to: D:\projects\f1-project\data\processed\pit_stops_2023_2024_first5.csv
Rows saved: 374


In [16]:
pit_stops_check = pd.read_csv(pit_stops_path)

print("Rows:", len(pit_stops_check))
print("Missing durations:", pit_stops_check["duration"].isna().sum())
print("Races:", pit_stops_check[["season", "round"]].drop_duplicates().shape[0])

Rows: 374
Missing durations: 70
Races: 10


In [17]:
# Check whether multiple pit-stop events share the same race, driver, and lap
same_lap_multiple_stops = (
    all_pit_stops_df
    .groupby(["season", "round", "driver_id", "lap"])
    .size()
    .reset_index(name="pit_stop_events")
)

same_lap_multiple_stops[
    same_lap_multiple_stops["pit_stop_events"] > 1
]

,season,round,driver_id,lap,pit_stop_events


In [18]:
# Keep only the keys needed to identify a pit-stop lap
pit_lap_keys = all_pit_stops_df[
    ["season", "round", "driver_id", "lap"]
].drop_duplicates()

# Mark pit-stop laps in the lap dataset
laps_with_pits_df = laps_df.merge(
    pit_lap_keys.assign(is_pit_lap=True),
    on=["season", "round", "driver_id", "lap"],
    how="left",
    validate="one_to_one"
)

# Laps without a matching pit-stop event are not pit laps
laps_with_pits_df["is_pit_lap"] = (
    laps_with_pits_df["is_pit_lap"].fillna(False)
)

print("Original lap rows:", len(laps_df))
print("Rows after merge:", len(laps_with_pits_df))
print("Pit-lap matches:", laps_with_pits_df["is_pit_lap"].sum())
print("Duplicate lap keys:", laps_with_pits_df.duplicated(
    subset=["season", "round", "driver_id", "lap"]
).sum())

Original lap rows: 10050
Rows after merge: 10050
Pit-lap matches: 374
Duplicate lap keys: 0


C:\Users\HP\AppData\Local\Temp\ipykernel_37588\4126428304.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  laps_with_pits_df["is_pit_lap"].fillna(False)


In [19]:
laps_with_pits_df["is_pit_lap"] = (
    laps_with_pits_df["is_pit_lap"]
    .astype("boolean")
    .fillna(False)
    .astype(bool)
)

In [20]:
print(laps_with_pits_df["is_pit_lap"].dtype)
print(laps_with_pits_df["is_pit_lap"].value_counts())

bool
is_pit_lap
False    9676
True      374
Name: count, dtype: int64


In [21]:
merged_path = processed_dir / "laps_with_pit_flags_2023_2024_first5.csv"

laps_with_pits_df.to_csv(merged_path, index=False)

print("Saved:", merged_path)

Saved: D:\projects\f1-project\data\processed\laps_with_pit_flags_2023_2024_first5.csv


In [22]:
race_audit_df = (
    laps_with_pits_df
    .groupby(["season", "round"])
    .agg(
        lap_records=("driver_id", "size"),
        drivers=("driver_id", "nunique"),
        first_lap=("lap", "min"),
        last_lap=("lap", "max"),
        pit_laps=("is_pit_lap", "sum")
    )
    .reset_index()
)

race_audit_df

,season,round,lap_records,drivers,first_lap,last_lap,pit_laps
0,2023,1,1055,20,1,57,50
1,2023,2,942,20,1,50,24
2,2023,3,995,19,1,58,65
3,2023,4,961,20,1,51,23
4,2023,5,1138,20,1,57,20
5,2024,1,1129,20,1,57,43
6,2024,2,900,20,1,50,19
7,2024,3,995,19,1,58,36
8,2024,4,905,18,1,53,54
9,2024,5,1030,20,1,56,40


In [29]:
from pathlib import Path
import requests
import pandas as pd

PIT_CACHE_DIR = Path("../data/raw/jolpica/pit_stops")
PIT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def fetch_race_pit_stops(season, round_num):
    cache_file = PIT_CACHE_DIR / f"{season}_round_{round_num}_pit_stops.json"

    if cache_file.exists():
        data = pd.read_json(cache_file, typ="series").to_dict()
    else:
        url = (
            f"https://api.jolpi.ca/ergast/f1/"
            f"{season}/{round_num}/pitstops/"
        )
        response = requests.get(url, params={"limit": 100}, timeout=30)
        response.raise_for_status()

        data = response.json()

        with open(cache_file, "w", encoding="utf-8") as f:
            import json
            json.dump(data, f)

    races = data["MRData"]["RaceTable"]["Races"]

    if not races:
        return pd.DataFrame()

    pit_stops = races[0].get("PitStops", [])

    rows = []
    for stop in pit_stops:
        rows.append({
            "season": int(season),
            "round": int(round_num),
            "driver_id": stop["driverId"],
            "lap": int(stop["lap"]),
            "stop": int(stop["stop"]),
            "time": stop.get("time"),
            "duration": pd.to_numeric(
                stop.get("duration"), errors="coerce"
            )
        })

    return pd.DataFrame(rows)


In [28]:
import json
from pathlib import Path
import requests
import pandas as pd

PIT_CACHE_DIR = Path("../data/raw/jolpica/pit_stops")
PIT_CACHE_DIR.mkdir(parents=True, exist_ok=True)

def fetch_race_pit_stops(season, round_num):
    cache_file = PIT_CACHE_DIR / f"{season}_round_{round_num}_pit_stops.json"

    # Load cached API response if it already exists
    if cache_file.exists():
        with open(cache_file, "r", encoding="utf-8") as f:
            data = json.load(f)

    # Otherwise, request data from the API and save it
    else:
        url = (
            f"https://api.jolpi.ca/ergast/f1/"
            f"{season}/{round_num}/pitstops/"
        )

        response = requests.get(
            url,
            params={"limit": 100},
            timeout=30
        )
        response.raise_for_status()

        data = response.json()

        with open(cache_file, "w", encoding="utf-8") as f:
            json.dump(data, f)

    races = data["MRData"]["RaceTable"]["Races"]

    if not races:
        return pd.DataFrame()

    pit_stops = races[0].get("PitStops", [])

    rows = []

    for stop in pit_stops:
        rows.append({
            "season": int(season),
            "round": int(round_num),
            "driver_id": stop["driverId"],
            "lap": int(stop["lap"]),
            "stop": int(stop["stop"]),
            "time": stop.get("time"),
            "duration": pd.to_numeric(
                stop.get("duration"),
                errors="coerce"
            )
        })

    return pd.DataFrame(rows)

In [30]:
test_pits_1 = fetch_race_pit_stops(2023, 1)
test_pits_2 = fetch_race_pit_stops(2023, 1)

test_pits_1.equals(test_pits_2)

True

In [31]:
# Get unique races from the lap dataset
race_list = (
    laps_df[["season", "round"]]
    .drop_duplicates()
    .sort_values(["season", "round"])
    .itertuples(index=False, name=None)
)

all_races = list(race_list)

len(all_races)

10

In [33]:
import requests
import pandas as pd

def get_season_schedule(season):
    url = f"https://api.jolpi.ca/ergast/f1/{season}.json"

    response = requests.get(
        url,
        params={"limit": 100},
        timeout=30
    )
    response.raise_for_status()

    data = response.json()
    races = data["MRData"]["RaceTable"]["Races"]

    schedule_rows = []

    for race in races:
        schedule_rows.append({
            "season": int(season),
            "round": int(race["round"]),
            "race_name": race["raceName"],
            "circuit": race["Circuit"]["circuitName"],
            "date": race["date"]
        })

    return pd.DataFrame(schedule_rows)

In [34]:
schedule_2023 = get_season_schedule(2023)
schedule_2024 = get_season_schedule(2024)

print("2023 races:", len(schedule_2023))
print("2024 races:", len(schedule_2024))

2023 races: 22
2024 races: 24


In [ ]:
# =========================
# 1. Imports
# =========================

import json
from pathlib import Path

import requests
import pandas as pd
import numpy as np


# =========================
# 2. Project paths
# =========================

# Relative to the notebooks/ folder
PROJECT_ROOT = Path("..")

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw" / "jolpica"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

LAP_CACHE_DIR = RAW_DATA_DIR / "laps"
PIT_CACHE_DIR = RAW_DATA_DIR / "pit_stops"

LAP_CACHE_DIR.mkdir(parents=True, exist_ok=True)
PIT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

API_BASE_URL = "https://api.jolpi.ca/ergast/f1/"


# =========================
# 3. Fetch lap data
# =========================

def fetch_race_laps(season, round_num, limit=100):
    """
    Fetch all lap-time records for one race from Jolpica.
    Uses pagination and caches each API page as JSON.
    """

    all_rows = []
    offset = 0

    while True:
        cache_file = (
            LAP_CACHE_DIR
            / f"{season}_round_{round_num}_laps_offset_{offset}.json"
        )

        # Load cached API response if available
        if cache_file.exists():
            with open(cache_file, "r", encoding="utf-8") as f:
                data = json.load(f)

        # Otherwise request the page and cache it
        else:
            url = (
                f"{API_BASE_URL}{season}/{round_num}/laps/"
            )

            response = requests.get(
                url,
                params={"limit": limit, "offset": offset},
                timeout=30
            )
            response.raise_for_status()

            data = response.json()

            with open(cache_file, "w", encoding="utf-8") as f:
                json.dump(data, f)

        mr_data = data["MRData"]
        races = mr_data["RaceTable"]["Races"]

        # No race data returned
        if not races:
            break

        lap_records = races[0].get("Laps", [])

        # Flatten nested lap and timing records
        for lap_record in lap_records:
            lap_number = int(lap_record["number"])

            for timing in lap_record.get("Timings", []):
                all_rows.append({
                    "season": int(season),
                    "round": int(round_num),
                    "lap": lap_number,
                    "driver_id": timing["driverId"],
                    "position": pd.to_numeric(
                        timing.get("position"),
                        errors="coerce"
                    ),
                    "lap_time": timing.get("time")
                })

        # Pagination metadata
        total = int(mr_data.get("total", 0))
        returned = len(lap_records)

        offset += returned

        # Stop when all lap records have been fetched
        if returned == 0 or offset >= total:
            break

    return pd.DataFrame(
        all_rows,
        columns=[
            "season", "round", "lap",
            "driver_id", "position", "lap_time"
        ]
    )


# =========================
# 4. Fetch season schedule
# =========================

def get_season_schedule(season):
    """
    Fetch the race schedule for a season.
    """

    cache_file = RAW_DATA_DIR / f"{season}_schedule.json"

    if cache_file.exists():
        with open(cache_file, "r", encoding="utf-8") as f:
            data = json.load(f)

    else:
        url = f"{API_BASE_URL}{season}.json"

        response = requests.get(
            url,
            params={"limit": 100},
            timeout=30
        )
        response.raise_for_status()

        data = response.json()

        with open(cache_file, "w", encoding="utf-8") as f:
            json.dump(data, f)

    races = data["MRData"]["RaceTable"]["Races"]

    schedule_rows = []

    for race in races:
        schedule_rows.append({
            "season": int(season),
            "round": int(race["round"]),
            "race_name": race["raceName"],
            "circuit": race["Circuit"]["circuitName"],
            "date": race["date"]
        })

    return pd.DataFrame(schedule_rows)


# =========================
# 5. Fetch pit-stop data
# =========================

def fetch_race_pit_stops(season, round_num):
    """
    Fetch pit-stop records for one race.
    Uses a local JSON cache.
    """

    cache_file = (
        PIT_CACHE_DIR
        / f"{season}_round_{round_num}_pit_stops.json"
    )

    if cache_file.exists():
        with open(cache_file, "r", encoding="utf-8") as f:
            data = json.load(f)

    else:
        url = (
            f"{API_BASE_URL}{season}/{round_num}/pitstops/"
        )
        
        response = requests.get(
            url,
            params={"limit": 100},
            timeout=30
        )
        response.raise_for_status()

        data = response.json()

        with open(cache_file, "w", encoding="utf-8") as f:
            json.dump(data, f)

    races = data["MRData"]["RaceTable"]["Races"]

    if not races:
        return pd.DataFrame(
            columns=[
                "season", "round", "driver_id",
                "lap", "stop", "time", "duration"
            ]
        )

    pit_stops = races[0].get("PitStops", [])

    rows = []

    for stop in pit_stops:
        rows.append({
            "season": int(season),
            "round": int(round_num),
            "driver_id": stop["driverId"],
            "lap": int(stop["lap"]),
            "stop": int(stop["stop"]),
            "time": stop.get("time"),
            "duration": pd.to_numeric(
                stop.get("duration"),
                errors="coerce"
            )
        })

    return pd.DataFrame(
        rows,
        columns=[
            "season", "round", "driver_id",
            "lap", "stop", "time", "duration"
        ]
    )


print("Setup complete.")
print("Lap cache:", LAP_CACHE_DIR.resolve())
print("Pit-stop cache:", PIT_CACHE_DIR.resolve())

Setup complete.
Lap cache: D:\projects\f1-project\data\raw\jolpica\laps
Pit-stop cache: D:\projects\f1-project\data\raw\jolpica\pit_stops


In [38]:
LAP_CACHE_DIR = RAW_DATA_DIR

In [39]:
test_laps = fetch_race_laps(2023, 1, limit=100)
test_pits = fetch_race_pit_stops(2023, 1)

print(test_laps.shape)
print(test_pits.shape)

HTTPError: 429 Client Error: Too Many Requests for url: https://api.jolpi.ca/ergast/f1/2023/1/laps/?limit=100&offset=632

In [40]:
from pathlib import Path

lap_cache_files = list(LAP_CACHE_DIR.glob("2023_round_1_laps_offset_*.json"))

print("Cached Bahrain lap pages:", len(lap_cache_files))
print([p.name for p in sorted(lap_cache_files)])

Cached Bahrain lap pages: 111
['2023_round_1_laps_offset_0.json', '2023_round_1_laps_offset_100.json', '2023_round_1_laps_offset_1000.json', '2023_round_1_laps_offset_101.json', '2023_round_1_laps_offset_107.json', '2023_round_1_laps_offset_11.json', '2023_round_1_laps_offset_113.json', '2023_round_1_laps_offset_119.json', '2023_round_1_laps_offset_125.json', '2023_round_1_laps_offset_131.json', '2023_round_1_laps_offset_137.json', '2023_round_1_laps_offset_143.json', '2023_round_1_laps_offset_149.json', '2023_round_1_laps_offset_155.json', '2023_round_1_laps_offset_161.json', '2023_round_1_laps_offset_167.json', '2023_round_1_laps_offset_17.json', '2023_round_1_laps_offset_173.json', '2023_round_1_laps_offset_179.json', '2023_round_1_laps_offset_185.json', '2023_round_1_laps_offset_191.json', '2023_round_1_laps_offset_197.json', '2023_round_1_laps_offset_200.json', '2023_round_1_laps_offset_203.json', '2023_round_1_laps_offset_209.json', '2023_round_1_laps_offset_215.json', '2023_roun

In [43]:
laps_df = pd.read_csv(
    "../data/processed/laps_2023_2024_first5.csv"
)

print("Rows:", len(laps_df))
print("Races:", laps_df[["season", "round"]].drop_duplicates().shape[0])
laps_df.head()

Rows: 10050
Races: 10


,season,round,lap,driver_id,position,lap_time
0,2023,1,1,max_verstappen,1,1:39.019
1,2023,1,1,leclerc,2,1:40.230
2,2023,1,1,perez,3,1:41.063
3,2023,1,1,sainz,4,1:41.659
4,2023,1,1,hamilton,5,1:42.288


In [44]:
existing_races = (
    laps_df[["season", "round"]]
    .drop_duplicates()
    .sort_values(["season", "round"])
    .reset_index(drop=True)
)

existing_races

,season,round
0,2023,1
1,2023,2
2,2023,3
3,2023,4
4,2023,5
5,2024,1
6,2024,2
7,2024,3
8,2024,4
9,2024,5


In [45]:
all_schedules = pd.concat(
    [schedule_2023, schedule_2024],
    ignore_index=True
)

all_race_keys = all_schedules[["season", "round"]].drop_duplicates()

missing_races = (
    all_race_keys
    .merge(
        existing_races,
        on=["season", "round"],
        how="left",
        indicator=True
    )
    .query("_merge == 'left_only'")
    .drop(columns="_merge")
    .sort_values(["season", "round"])
    .reset_index(drop=True)
)

print("Total scheduled races:", len(all_race_keys))
print("Already collected:", len(existing_races))
print("Missing races:", len(missing_races))

missing_races.head(10)

Total scheduled races: 46
Already collected: 10
Missing races: 36


,season,round
0,2023,6
1,2023,7
2,2023,8
3,2023,9
4,2023,10
5,2023,11
6,2023,12
7,2023,13
8,2023,14
9,2023,15


In [47]:
from pathlib import Path

print("Lap dataset exists:", Path("../data/processed/laps_2023_2024_first5.csv").exists())
print("Pit-stop dataset exists:", Path("../data/processed/pit_stops_2023_2024_first5.csv").exists())
print("Merged dataset exists:", Path("../data/processed/laps_with_pit_flags_2023_2024_first5.csv").exists())

Lap dataset exists: True
Pit-stop dataset exists: True
Merged dataset exists: True


In [48]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("../data/processed")

laps = pd.read_csv(DATA_DIR / "laps_with_pit_flags_2023_2024_first5.csv")

print("Shape:", laps.shape)
print("Columns:", laps.columns.tolist())
laps.head()

Shape: (10050, 7)
Columns: ['season', 'round', 'lap', 'driver_id', 'position', 'lap_time', 'is_pit_lap']


,season,round,lap,driver_id,position,lap_time,is_pit_lap
0,2023,1,1,max_verstappen,1,1:39.019,False
1,2023,1,1,leclerc,2,1:40.230,False
2,2023,1,1,perez,3,1:41.063,False
3,2023,1,1,sainz,4,1:41.659,False
4,2023,1,1,hamilton,5,1:42.288,False


In [49]:
def lap_time_to_seconds(value):
    if pd.isna(value):
        return None

    value = str(value).strip()

    try:
        minutes, seconds = value.split(":")
        return int(minutes) * 60 + float(seconds)
    except ValueError:
        return None


laps["lap_time_seconds"] = laps["lap_time"].apply(lap_time_to_seconds)

print("Missing lap times:", laps["lap_time_seconds"].isna().sum())
laps[["lap_time", "lap_time_seconds"]].head(10)

Missing lap times: 0


,lap_time,lap_time_seconds
0,1:39.019,99.019
1,1:40.230,100.230
2,1:41.063,101.063
3,1:41.659,101.659
4,1:42.288,102.288
5,1:42.662,102.662
6,1:43.608,103.608
7,1:44.154,104.154
8,1:44.670,104.670
9,1:45.364,105.364


In [50]:
# Keep plausible race-lap times for this first exploratory pass.
# This is a broad filter, not a claim that every retained lap is clean.
pace_df = laps[
    laps["lap_time_seconds"].between(50, 200)
].copy()

print("Rows before filtering:", len(laps))
print("Rows after broad time filter:", len(pace_df))
print("Rows removed:", len(laps) - len(pace_df))

print("\nPit-lap counts:")
print(pace_df["is_pit_lap"].value_counts())

print("\nLap-time summary:")
display(pace_df["lap_time_seconds"].describe())

Rows before filtering: 10050
Rows after broad time filter: 9966
Rows removed: 84

Pit-lap counts:
is_pit_lap
False    9636
True      330
Name: count, dtype: int64

Lap-time summary:


count    9966.000000
mean       98.323130
std        13.705911
min        79.813000
25%        92.903250
50%        96.614500
75%       100.891250
max       196.801000
Name: lap_time_seconds, dtype: float64

In [51]:
# Exclude recorded pit-stop laps for the initial pace baseline.
clean_laps = pace_df[pace_df["is_pit_lap"] == False].copy()

print("Candidate clean laps:", len(clean_laps))
print("Unique races:", clean_laps[["season", "round"]].drop_duplicates().shape[0])
print("Unique drivers:", clean_laps["driver_id"].nunique())

display(
    clean_laps.groupby(["season", "round"])["lap_time_seconds"]
    .agg(["count", "median", "mean"])
    .reset_index()
    .head(10)
)

Candidate clean laps: 9636
Unique races: 10
Unique drivers: 22


,season,round,count,median,mean
0,2023,1,1005,98.4870,100.279741
1,2023,2,918,95.3235,98.428587
2,2023,3,909,83.0080,88.931012
3,2023,4,938,107.3420,110.144333
4,2023,5,1118,92.8290,93.239624
5,2024,1,1086,96.9760,97.959228
6,2024,2,881,94.3780,97.730841
7,2024,3,959,82.7970,83.934260
8,2024,4,833,97.5110,99.425989
9,2024,5,989,101.9980,108.560923


In [52]:
# Median lap time per driver per race
driver_race_pace = (
    clean_laps
    .groupby(["season", "round", "driver_id"])["lap_time_seconds"]
    .median()
    .reset_index(name="median_lap_time")
)

print("Driver-race records:", len(driver_race_pace))
display(driver_race_pace.head(15))

Driver-race records: 196


,season,round,driver_id,median_lap_time
0,2023,1,albon,98.3380
1,2023,1,alonso,98.1240
2,2023,1,bottas,98.4940
3,2023,1,de_vries,99.1820
4,2023,1,gasly,98.3605
5,2023,1,hamilton,98.3390
6,2023,1,hulkenberg,98.9720
7,2023,1,kevin_magnussen,98.9130
8,2023,1,leclerc,98.5720
9,2023,1,max_verstappen,97.6480


In [ ]:
# Fastest and slowest median pace within each race
race_pace_summary = (
    driver_race_pace
    .groupby(["season", "round"])["median_lap_time"]
    .agg(["min", "median", "max"])
    .reset_index()
)

display(race_pace_summary)

,season,round,min,median,max
0,2023,1,97.4540,98.46600,101.1670
1,2023,2,93.4260,95.50025,96.9840
2,2023,3,81.9980,82.61300,95.3500
3,2023,4,105.5220,107.62025,109.3370
4,2023,5,91.6975,92.87350,93.5280
5,2024,1,95.6000,97.09500,97.9755
6,2024,2,92.8890,94.92050,121.4530
7,2024,3,81.5425,82.96800,84.0990
8,2024,4,96.3910,97.47350,100.1095
9,2024,5,100.0275,101.96625,103.8205


: 

In [1]:
from pathlib import Path
import pandas as pd

root = Path(r"D:\projects\f1-project")

for p in sorted(root.rglob("*")):
    if p.is_file() and ".git" not in p.parts and p.suffix in {".csv", ".py", ".ipynb", ".md"}:
        print(p.relative_to(root))

print("cached JSON files:", len(list(root.rglob("*.json"))))

for p in (root / "data").rglob("*.csv"):
    df = pd.read_csv(p)
    print("\n", p.name, df.shape)
    print(list(df.columns))

.venv\Lib\site-packages\_argon2_cffi_bindings\__init__.py
.venv\Lib\site-packages\_distutils_hack\__init__.py
.venv\Lib\site-packages\_distutils_hack\override.py
.venv\Lib\site-packages\_yaml\__init__.py
.venv\Lib\site-packages\altair\__init__.py
.venv\Lib\site-packages\altair\_magics.py
.venv\Lib\site-packages\altair\datasets\__init__.py
.venv\Lib\site-packages\altair\datasets\_cache.py
.venv\Lib\site-packages\altair\datasets\_constraints.py
.venv\Lib\site-packages\altair\datasets\_data.py
.venv\Lib\site-packages\altair\datasets\_exceptions.py
.venv\Lib\site-packages\altair\datasets\_loader.py
.venv\Lib\site-packages\altair\datasets\_reader.py
.venv\Lib\site-packages\altair\datasets\_readimpl.py
.venv\Lib\site-packages\altair\datasets\_typing.py
.venv\Lib\site-packages\altair\expr\__init__.py
.venv\Lib\site-packages\altair\expr\consts.py
.venv\Lib\site-packages\altair\expr\core.py
.venv\Lib\site-packages\altair\expr\funcs.py
.venv\Lib\site-packages\altair\jupyter\__init__.py
.venv\Lib